In [3]:
# ============================================================
# PHASE 2 — DATA QUALITY AUDIT
# ============================================================
# Purpose:
# Diagnose and classify the data-quality issues discovered
# during Phase 1.
#
# IMPORTANT:
# - The raw dataset is NOT modified.
# - Numeric/date conversions are temporary and audit-only.
# - This phase identifies problems; it does not clean them.
# - The final output is a structured data-quality report.
#
# Business question:
# "Which observations represent genuine data-quality problems,
# how severe are they, and what should we do about them?"
# ============================================================


# ============================================================
# 1. IMPORT LIBRARIES
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path


# ============================================================
# 2. LOAD RAW DATASET
# ============================================================
# Keep the source data completely untouched.

file_path = Path(
    r"C:\Users\hp\Downloads\Working_capital_dirty .csv"
)

df = pd.read_csv(file_path)

print("=" * 70)
print("PHASE 2 — DATA QUALITY AUDIT")
print("=" * 70)

print(f"\nDataset shape: {df.shape}")
print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns)}")


# ============================================================
# 3. AUDIT CONFIGURATION
# ============================================================

KEY_COLUMNS = [
    "invoice_id",
    "customer_id",
    "supplier_id",
    "product_id"
]

NUMERIC_COLUMNS = [
    "invoice_amount",
    "payment_amount"
]

DATE_COLUMNS = [
    "invoice_date",
    "due_date",
    "payment_date"
]

EXPECTED_STATUS_VALUES = {
    "Paid",
    "Open"
}

EXPECTED_CURRENCY_VALUES = {
    "MAD",
    "EUR"
}

EXPECTED_PAYMENT_METHOD_VALUES = {
    "Bank Transfer",
    "Cheque",
    "Credit Card",
    "Cash"
}


# ============================================================
# 4. TEMPORARY STANDARDIZED VIEWS FOR AUDITING
# ============================================================
# IMPORTANT:
# These are NOT cleaning operations.
#
# We create temporary versions only so the audit can correctly
# interpret values such as:
#
# "9,501.81"  -> 9501.81
# "85,000.00" -> 85000.00
#
# The raw dataframe remains unchanged.


invoice_amount_numeric = pd.to_numeric(
    df["invoice_amount"]
    .astype("string")
    .str.replace(",", "", regex=False)
    .str.strip(),
    errors="coerce"
)

payment_amount_numeric = pd.to_numeric(
    df["payment_amount"],
    errors="coerce"
)

invoice_dates = pd.to_datetime(
    df["invoice_date"],
    errors="coerce"
)

due_dates = pd.to_datetime(
    df["due_date"],
    errors="coerce"
)

payment_dates = pd.to_datetime(
    df["payment_date"],
    errors="coerce"
)


# ============================================================
# 5. STRUCTURED ISSUE REPORT
# ============================================================
# Every finding is stored as a row.
#
# This gives us a reusable data-quality artifact rather than
# relying only on printed output.

issues = []


def add_issue(
    issue_id,
    category,
    rule,
    count,
    severity,
    interpretation,
    recommended_action
):
    percentage = (
        count / len(df) * 100
        if len(df) > 0
        else 0
    )

    issues.append({
        "issue_id": issue_id,
        "category": category,
        "rule": rule,
        "affected_rows": int(count),
        "affected_pct": round(percentage, 2),
        "severity": severity,
        "interpretation": interpretation,
        "recommended_action": recommended_action
    })


# ============================================================
# 6. MISSING-VALUE AUDIT
# ============================================================

print("\n" + "=" * 70)
print("6. MISSING-VALUE AUDIT")
print("=" * 70)


# ------------------------------------------------------------
# 6.1 Open invoices without payment information
# ------------------------------------------------------------
# This is expected behavior for genuinely unpaid invoices.

open_mask = (
    df["status"]
    .astype("string")
    .str.strip()
    .eq("Open")
)

open_missing_payment_date = (
    open_mask &
    df["payment_date"].isna()
)

open_missing_payment_amount = (
    open_mask &
    df["payment_amount"].isna()
)

count_open_missing_payment_date = (
    open_missing_payment_date.sum()
)

count_open_missing_payment_amount = (
    open_missing_payment_amount.sum()
)

print(
    f"\nOpen invoices missing payment_date: "
    f"{count_open_missing_payment_date:,}"
)

print(
    f"Open invoices missing payment_amount: "
    f"{count_open_missing_payment_amount:,}"
)

add_issue(
    "MISSING_001",
    "Missing Values",
    "Open invoices may legitimately have no payment_date",
    count_open_missing_payment_date,
    "EXPECTED",
    "An unpaid/open invoice normally has no payment date.",
    "Keep missing payment_date for genuinely open invoices."
)

add_issue(
    "MISSING_002",
    "Missing Values",
    "Open invoices may legitimately have no payment_amount",
    count_open_missing_payment_amount,
    "EXPECTED",
    "An unpaid/open invoice normally has no recorded payment amount.",
    "Keep missing payment_amount for genuinely open invoices."
)


# ------------------------------------------------------------
# 6.2 Paid invoices missing payment information
# ------------------------------------------------------------

paid_mask = (
    df["status"]
    .astype("string")
    .str.strip()
    .eq("Paid")
)

paid_missing_payment_date = (
    paid_mask &
    df["payment_date"].isna()
)

paid_missing_payment_amount = (
    paid_mask &
    df["payment_amount"].isna()
)

count_paid_missing_payment_date = (
    paid_missing_payment_date.sum()
)

count_paid_missing_payment_amount = (
    paid_missing_payment_amount.sum()
)

print(
    f"Paid invoices missing payment_date: "
    f"{count_paid_missing_payment_date:,}"
)

print(
    f"Paid invoices missing payment_amount: "
    f"{count_paid_missing_payment_amount:,}"
)

add_issue(
    "MISSING_003",
    "Missing Values",
    "Paid invoices should contain payment_date",
    count_paid_missing_payment_date,
    "CRITICAL",
    "A Paid status without a payment date conflicts with the expected payment lifecycle.",
    "Investigate against source payment records."
)

add_issue(
    "MISSING_004",
    "Missing Values",
    "Paid invoices should contain payment_amount",
    count_paid_missing_payment_amount,
    "HIGH",
    "A Paid status without a payment amount prevents reliable payment reconciliation.",
    "Investigate and recover the payment amount where possible."
)


# ------------------------------------------------------------
# 6.3 Missing invoice amount
# ------------------------------------------------------------

missing_invoice_amount = df["invoice_amount"].isna()

count_missing_invoice_amount = (
    missing_invoice_amount.sum()
)

print(
    f"Missing invoice_amount: "
    f"{count_missing_invoice_amount:,}"
)

add_issue(
    "MISSING_005",
    "Missing Values",
    "Invoice amount is required for financial analysis",
    count_missing_invoice_amount,
    "CRITICAL",
    "Without invoice amount, transaction-level financial calculations cannot be trusted.",
    "Investigate source data and recover the amount if possible."
)


# ------------------------------------------------------------
# 6.4 Missing customer ID
# ------------------------------------------------------------

missing_customer_id = df["customer_id"].isna()

count_missing_customer_id = (
    missing_customer_id.sum()
)

print(
    f"Missing customer_id: "
    f"{count_missing_customer_id:,}"
)

add_issue(
    "MISSING_006",
    "Missing Values",
    "Customer identifier should be present",
    count_missing_customer_id,
    "HIGH",
    "Missing customer identifiers prevent reliable customer-level analysis.",
    "Recover customer_id from source/customer master where possible."
)


# ------------------------------------------------------------
# 6.5 Missing region
# ------------------------------------------------------------

missing_region = df["region"].isna()

count_missing_region = (
    missing_region.sum()
)

print(
    f"Missing region: "
    f"{count_missing_region:,}"
)

add_issue(
    "MISSING_007",
    "Missing Values",
    "Region should be populated",
    count_missing_region,
    "MEDIUM",
    "Missing region reduces the reliability of geographic analysis.",
    "Recover region from customer/master data where possible."
)


# ------------------------------------------------------------
# 6.6 Missing payment method
# ------------------------------------------------------------

missing_payment_method = (
    df["payment_method"].isna()
)

count_missing_payment_method = (
    missing_payment_method.sum()
)

print(
    f"Missing payment_method: "
    f"{count_missing_payment_method:,}"
)

add_issue(
    "MISSING_008",
    "Missing Values",
    "Payment method should be populated",
    count_missing_payment_method,
    "MEDIUM",
    "Missing payment method limits payment-behavior analysis.",
    "Recover from payment records where possible."
)


# ============================================================
# 7. DUPLICATE-RECORD AUDIT
# ============================================================
# We distinguish:
#
# 1. Duplicate invoice IDs
# 2. Rows participating in duplicate groups
# 3. Excess duplicate rows
#
# This is important because "219 duplicate rows" and
# "110 excess records" mean different things.


print("\n" + "=" * 70)
print("7. DUPLICATE-RECORD AUDIT")
print("=" * 70)


# ------------------------------------------------------------
# 7.1 Exact duplicate groups
# ------------------------------------------------------------

exact_duplicate_mask = df.duplicated(
    keep=False
)

rows_in_exact_duplicate_groups = (
    exact_duplicate_mask.sum()
)

excess_exact_duplicate_rows = (
    df.duplicated(keep="first").sum()
)

print(
    f"\nRows belonging to exact duplicate groups: "
    f"{rows_in_exact_duplicate_groups:,}"
)

print(
    f"Excess exact duplicate rows: "
    f"{excess_exact_duplicate_rows:,}"
)

add_issue(
    "DUP_001",
    "Duplicates",
    "Exact duplicate records should not exist at invoice grain",
    excess_exact_duplicate_rows,
    "HIGH",
    "Repeated copies of the same transaction can double-count financial metrics.",
    "Remove excess duplicate copies during Phase 3 after confirming invoice grain."
)


# ------------------------------------------------------------
# 7.2 Duplicate invoice IDs
# ------------------------------------------------------------

duplicate_invoice_mask = (
    df["invoice_id"].duplicated(keep=False)
)

duplicate_invoice_ids = (
    df.loc[
        duplicate_invoice_mask,
        "invoice_id"
    ].nunique()
)

duplicate_invoice_rows = (
    duplicate_invoice_mask.sum()
)

duplicate_invoice_excess_rows = (
    df["invoice_id"].duplicated(keep="first").sum()
)

print(
    f"Unique duplicated invoice IDs: "
    f"{duplicate_invoice_ids:,}"
)

print(
    f"Rows associated with duplicated invoice IDs: "
    f"{duplicate_invoice_rows:,}"
)

print(
    f"Excess duplicate invoice rows: "
    f"{duplicate_invoice_excess_rows:,}"
)

add_issue(
    "DUP_002",
    "Duplicates",
    "invoice_id should identify one invoice",
    duplicate_invoice_excess_rows,
    "HIGH",
    "Repeated invoice IDs violate the expected one-row-per-invoice grain.",
    "Resolve excess duplicate records during Phase 3."
)


# ============================================================
# 8. KEY-INTEGRITY AUDIT
# ============================================================

print("\n" + "=" * 70)
print("8. KEY-INTEGRITY AUDIT")
print("=" * 70)


# ------------------------------------------------------------
# invoice_id
# ------------------------------------------------------------

invoice_id_missing = (
    df["invoice_id"].isna().sum()
)

invoice_id_duplicate_excess = (
    df["invoice_id"].duplicated(keep="first").sum()
)

print(
    f"\nMissing invoice_id: "
    f"{invoice_id_missing:,}"
)

print(
    f"Excess duplicate invoice_id rows: "
    f"{invoice_id_duplicate_excess:,}"
)

add_issue(
    "KEY_001",
    "Key Integrity",
    "invoice_id must be populated",
    invoice_id_missing,
    "CRITICAL",
    "invoice_id is the expected transaction identifier.",
    "Investigate and recover missing invoice identifiers."
)

add_issue(
    "KEY_002",
    "Key Integrity",
    "invoice_id should be unique",
    invoice_id_duplicate_excess,
    "HIGH",
    "Duplicate invoice IDs can cause double counting at invoice grain.",
    "Resolve duplicates during Phase 3."
)


# ------------------------------------------------------------
# supplier_id
# ------------------------------------------------------------

missing_supplier_id = (
    df["supplier_id"].isna().sum()
)

print(
    f"Missing supplier_id: "
    f"{missing_supplier_id:,}"
)

add_issue(
    "KEY_003",
    "Key Integrity",
    "supplier_id should be populated",
    missing_supplier_id,
    "HIGH",
    "Missing supplier identifiers weaken supplier-level analysis.",
    "Recover supplier_id from source/master data where possible."
)


# ------------------------------------------------------------
# product_id
# ------------------------------------------------------------

missing_product_id = (
    df["product_id"].isna().sum()
)

print(
    f"Missing product_id: "
    f"{missing_product_id:,}"
)

add_issue(
    "KEY_004",
    "Key Integrity",
    "product_id should be populated",
    missing_product_id,
    "HIGH",
    "Missing product identifiers weaken product-level analysis.",
    "Recover product_id from source/master data where possible."
)


# ============================================================
# 9. NUMERIC-INTEGRITY AUDIT
# ============================================================
# IMPORTANT:
# invoice_amount contains thousands separators such as
# "9,501.81".
#
# We remove commas ONLY in the temporary audit representation.
# The raw dataframe is untouched.


print("\n" + "=" * 70)
print("9. NUMERIC-INTEGRITY AUDIT")
print("=" * 70)


# ------------------------------------------------------------
# 9.1 Non-numeric invoice amounts
# ------------------------------------------------------------

non_numeric_invoice_amount = (
    df["invoice_amount"].notna() &
    invoice_amount_numeric.isna()
)

count_non_numeric_invoice_amount = (
    non_numeric_invoice_amount.sum()
)

print(
    f"\nPresent but non-numeric invoice_amount values: "
    f"{count_non_numeric_invoice_amount:,}"
)

add_issue(
    "NUM_001",
    "Numeric Integrity",
    "invoice_amount must contain a valid numeric value",
    count_non_numeric_invoice_amount,
    "CRITICAL",
    "Text placeholders or invalid strings cannot be used reliably in financial calculations.",
    "Standardize valid numeric formatting and investigate invalid placeholders during Phase 3."
)


# ------------------------------------------------------------
# 9.2 Negative invoice amounts
# ------------------------------------------------------------

negative_invoice_amount = (
    invoice_amount_numeric < 0
)

count_negative_invoice_amount = (
    negative_invoice_amount.sum()
)

print(
    f"Negative invoice amounts: "
    f"{count_negative_invoice_amount:,}"
)

add_issue(
    "NUM_002",
    "Numeric Integrity",
    "Negative invoice amounts require business interpretation",
    count_negative_invoice_amount,
    "INVESTIGATE",
    "Negative invoices may represent credit notes, refunds, reversals, or accounting adjustments.",
    "Confirm the accounting meaning before modifying or excluding them."
)


# ------------------------------------------------------------
# 9.3 Negative payment amounts
# ------------------------------------------------------------

negative_payment_amount = (
    payment_amount_numeric < 0
)

count_negative_payment_amount = (
    negative_payment_amount.sum()
)

print(
    f"Negative payment amounts: "
    f"{count_negative_payment_amount:,}"
)

add_issue(
    "NUM_003",
    "Numeric Integrity",
    "Negative payment amounts require business interpretation",
    count_negative_payment_amount,
    "INVESTIGATE",
    "Negative payments may represent reversals, refunds, or corrections.",
    "Confirm accounting treatment before modifying."
)


# ------------------------------------------------------------
# 9.4 Zero payment amounts
# ------------------------------------------------------------

zero_payment_amount = (
    payment_amount_numeric == 0
)

count_zero_payment_amount = (
    zero_payment_amount.sum()
)

print(
    f"Zero payment amounts: "
    f"{count_zero_payment_amount:,}"
)

add_issue(
    "NUM_004",
    "Numeric Integrity",
    "Zero payment amounts require business interpretation",
    count_zero_payment_amount,
    "INVESTIGATE",
    "A zero payment may represent an unusual transaction, correction, or incomplete record.",
    "Check status and transaction context before deciding whether it is valid."
)


# ============================================================
# 10. DATE-INTEGRITY AUDIT
# ============================================================

print("\n" + "=" * 70)
print("10. DATE-INTEGRITY AUDIT")
print("=" * 70)


for column, converted in {
    "invoice_date": invoice_dates,
    "due_date": due_dates,
    "payment_date": payment_dates
}.items():

    missing_count = df[column].isna().sum()

    invalid_present = (
        df[column].notna() &
        converted.isna()
    )

    invalid_count = invalid_present.sum()

    print(f"\n{column}")
    print(f"  Missing: {missing_count:,}")
    print(f"  Present but invalid: {invalid_count:,}")

    add_issue(
        f"DATE_{column.upper()}",
        "Date Integrity",
        f"{column} must contain a valid date when populated",
        invalid_count,
        "HIGH",
        "Invalid date strings cannot support reliable time-based analysis.",
        f"Standardize valid {column} values and investigate invalid records."
    )


# ------------------------------------------------------------
# 10.1 Date sequence rules
# ------------------------------------------------------------

due_before_invoice = (
    due_dates.notna() &
    invoice_dates.notna() &
    (due_dates < invoice_dates)
)

payment_before_invoice = (
    payment_dates.notna() &
    invoice_dates.notna() &
    (payment_dates < invoice_dates)
)

payment_before_due = (
    payment_dates.notna() &
    due_dates.notna() &
    (payment_dates < due_dates)
)

count_due_before_invoice = (
    due_before_invoice.sum()
)

count_payment_before_invoice = (
    payment_before_invoice.sum()
)

count_payment_before_due = (
    payment_before_due.sum()
)

print(
    f"\nDue date before invoice date: "
    f"{count_due_before_invoice:,}"
)

print(
    f"Payment date before invoice date: "
    f"{count_payment_before_invoice:,}"
)

print(
    f"Payment date before due date: "
    f"{count_payment_before_due:,}"
)

add_issue(
    "DATE_004",
    "Date Integrity",
    "due_date should not precede invoice_date",
    count_due_before_invoice,
    "CRITICAL",
    "A due date earlier than the invoice date contradicts the expected invoice lifecycle.",
    "Investigate and correct against source documentation."
)

add_issue(
    "DATE_005",
    "Date Integrity",
    "payment_date should not precede invoice_date",
    count_payment_before_invoice,
    "CRITICAL",
    "A payment normally cannot occur before the invoice exists.",
    "Investigate transaction dates."
)

add_issue(
    "DATE_006",
    "Date Integrity",
    "Payment before due date requires interpretation",
    count_payment_before_due,
    "INVESTIGATE",
    "Early payment may be legitimate and can indicate customers paying before the contractual due date.",
    "Keep unless source business rules prohibit early payment."
)


# ============================================================
# 11. CATEGORICAL-CONSISTENCY AUDIT
# ============================================================

print("\n" + "=" * 70)
print("11. CATEGORICAL-CONSISTENCY AUDIT")
print("=" * 70)


# ------------------------------------------------------------
# 11.1 Region
# ------------------------------------------------------------

region_trimmed = (
    df["region"]
    .astype("string")
    .str.strip()
)

region_title = (
    region_trimmed
    .str.title()
)

region_standard_map = {
    "Marrakech": "Marrakesh",
    "Tanger": "Tangier"
}

region_standard_candidate = (
    region_title
    .replace(region_standard_map)
)

region_nonstandard = (
    df["region"].notna() &
    (
        region_standard_candidate
        != df["region"].astype("string").str.strip()
    )
)

count_region_nonstandard = (
    region_nonstandard.sum()
)

print(
    f"\nRegion representation inconsistencies: "
    f"{count_region_nonstandard:,}"
)

add_issue(
    "CAT_001",
    "Categorical Consistency",
    "Region values should use one standardized representation",
    count_region_nonstandard,
    "MEDIUM",
    "Different spelling, capitalization, or whitespace can split the same region into multiple analytical categories.",
    "Apply an explicit region mapping during Phase 3."
)


# ------------------------------------------------------------
# 11.2 Currency
# ------------------------------------------------------------

currency_trimmed = (
    df["currency"]
    .astype("string")
    .str.strip()
)

currency_standard_candidate = (
    currency_trimmed.str.upper()
)

currency_nonstandard = (
    df["currency"].notna() &
    (
        currency_standard_candidate
        != df["currency"].astype("string")
    )
)

count_currency_nonstandard = (
    currency_nonstandard.sum()
)

print(
    f"Currency formatting inconsistencies: "
    f"{count_currency_nonstandard:,}"
)

add_issue(
    "CAT_002",
    "Categorical Consistency",
    "Currency codes should use standardized formatting",
    count_currency_nonstandard,
    "MEDIUM",
    "Whitespace and inconsistent formatting can fragment currency analysis.",
    "Trim whitespace and standardize currency codes."
)


# ------------------------------------------------------------
# 11.3 Payment method
# ------------------------------------------------------------

payment_method_trimmed = (
    df["payment_method"]
    .astype("string")
    .str.strip()
)

payment_method_standard_candidate = (
    payment_method_trimmed
    .str.title()
)

payment_method_nonstandard = (
    df["payment_method"].notna() &
    (
        payment_method_standard_candidate
        != df["payment_method"].astype("string")
    )
)

count_payment_method_nonstandard = (
    payment_method_nonstandard.sum()
)

print(
    f"Payment method representation inconsistencies: "
    f"{count_payment_method_nonstandard:,}"
)

add_issue(
    "CAT_003",
    "Categorical Consistency",
    "Payment method values should use standardized representation",
    count_payment_method_nonstandard,
    "MEDIUM",
    "Capitalization differences can split one payment method into multiple categories.",
    "Standardize payment method labels."
)


# ============================================================
# 12. UNEXPECTED CATEGORY VALUES
# ============================================================

print("\n" + "=" * 70)
print("12. UNEXPECTED CATEGORY VALUES")
print("=" * 70)


# ------------------------------------------------------------
# Status
# ------------------------------------------------------------

status_values = (
    df["status"]
    .dropna()
    .astype("string")
    .str.strip()
)

unexpected_status_count = (
    ~status_values.isin(
        EXPECTED_STATUS_VALUES
    )
).sum()

print(
    f"\nUnexpected status values: "
    f"{unexpected_status_count:,}"
)

add_issue(
    "CAT_004",
    "Categorical Consistency",
    "Status must belong to the expected business domain",
    unexpected_status_count,
    "CRITICAL",
    "Unexpected status values can break payment lifecycle logic.",
    "Validate against the source system's status definitions."
)


# ------------------------------------------------------------
# Currency
# ------------------------------------------------------------

currency_values = (
    df["currency"]
    .dropna()
    .astype("string")
    .str.strip()
    .str.upper()
)

unexpected_currency_count = (
    ~currency_values.isin(
        EXPECTED_CURRENCY_VALUES
    )
).sum()

print(
    f"Unexpected currency values: "
    f"{unexpected_currency_count:,}"
)

add_issue(
    "CAT_005",
    "Categorical Consistency",
    "Currency must belong to the supported currency domain",
    unexpected_currency_count,
    "HIGH",
    "Unsupported currencies can make financial aggregation invalid unless FX conversion is handled.",
    "Validate currency coverage and define FX treatment."
)


# ------------------------------------------------------------
# Payment method
# ------------------------------------------------------------

payment_method_values = (
    df["payment_method"]
    .dropna()
    .astype("string")
    .str.strip()
    .str.title()
)

unexpected_payment_method_count = (
    ~payment_method_values.isin(
        EXPECTED_PAYMENT_METHOD_VALUES
    )
).sum()

print(
    f"Unexpected payment method values: "
    f"{unexpected_payment_method_count:,}"
)

add_issue(
    "CAT_006",
    "Categorical Consistency",
    "Payment method must belong to the supported domain",
    unexpected_payment_method_count,
    "MEDIUM",
    "Unexpected payment methods can fragment payment-method analysis.",
    "Validate against the payment system's allowed values."
)


# ============================================================
# 13. BUSINESS-RULE AUDIT
# ============================================================

print("\n" + "=" * 70)
print("13. BUSINESS-RULE AUDIT")
print("=" * 70)


# ------------------------------------------------------------
# 13.1 Paid invoice without payment evidence
# ------------------------------------------------------------

paid_without_payment = (
    paid_mask &
    (
        df["payment_date"].isna() |
        df["payment_amount"].isna()
    )
)

count_paid_without_payment = (
    paid_without_payment.sum()
)

print(
    f"\nPaid invoices missing payment evidence: "
    f"{count_paid_without_payment:,}"
)

add_issue(
    "BUS_001",
    "Business Rules",
    "Paid invoices should have payment evidence",
    count_paid_without_payment,
    "CRITICAL",
    "Paid status conflicts with missing payment information.",
    "Investigate payment records and source-system status."
)


# ------------------------------------------------------------
# 13.2 Open invoice with payment evidence
# ------------------------------------------------------------

open_with_payment = (
    open_mask &
    (
        df["payment_date"].notna() |
        df["payment_amount"].notna()
    )
)

count_open_with_payment = (
    open_with_payment.sum()
)

print(
    f"Open invoices with payment evidence: "
    f"{count_open_with_payment:,}"
)

add_issue(
    "BUS_002",
    "Business Rules",
    "Open invoices with payment evidence require review",
    count_open_with_payment,
    "HIGH",
    "An Open status alongside payment information may indicate a stale status or partial-payment behavior.",
    "Determine whether Open means unpaid, partially paid, or outstanding."
)


# ------------------------------------------------------------
# 13.3 Payment greater than invoice
# ------------------------------------------------------------
# This is a FLAG, not proof of corruption.
#
# Possible explanations include:
# - overpayment
# - adjustment
# - duplicate payment
# - refund/reversal
# - currency issue
# - source-data error


payment_gt_invoice = (
    payment_amount_numeric.notna() &
    invoice_amount_numeric.notna() &
    (
        payment_amount_numeric >
        invoice_amount_numeric
    )
)

count_payment_gt_invoice = (
    payment_gt_invoice.sum()
)

print(
    f"Payment amount greater than invoice amount: "
    f"{count_payment_gt_invoice:,}"
)

add_issue(
    "BUS_003",
    "Business Rules",
    "Payment amount should be reconciled against invoice amount",
    count_payment_gt_invoice,
    "HIGH",
    "Payment exceeding invoice amount may represent an overpayment, adjustment, duplicate payment, currency issue, or data error.",
    "Investigate before using payment_amount for receivables calculations."
)


# ============================================================
# 14. FINANCIAL RECONCILIATION AUDIT
# ============================================================

print("\n" + "=" * 70)
print("14. FINANCIAL RECONCILIATION AUDIT")
print("=" * 70)


# ------------------------------------------------------------
# 14.1 Payment amount without payment date
# ------------------------------------------------------------

payment_amount_without_date = (
    df["payment_amount"].notna() &
    df["payment_date"].isna()
)

count_payment_amount_without_date = (
    payment_amount_without_date.sum()
)

print(
    f"\nPayment amount present but payment date missing: "
    f"{count_payment_amount_without_date:,}"
)

add_issue(
    "FIN_001",
    "Financial Reconciliation",
    "Payment amount should normally have a payment date",
    count_payment_amount_without_date,
    "HIGH",
    "A payment amount without a payment date weakens payment-timing analysis.",
    "Investigate transaction history."
)


# ------------------------------------------------------------
# 14.2 Payment date without payment amount
# ------------------------------------------------------------

payment_date_without_amount = (
    df["payment_date"].notna() &
    df["payment_amount"].isna()
)

count_payment_date_without_amount = (
    payment_date_without_amount.sum()
)

print(
    f"Payment date present but payment amount missing: "
    f"{count_payment_date_without_amount:,}"
)

add_issue(
    "FIN_002",
    "Financial Reconciliation",
    "Payment date should normally have a payment amount",
    count_payment_date_without_amount,
    "HIGH",
    "A payment date without an amount provides timing information but cannot support complete payment reconciliation.",
    "Recover payment amount where possible."
)


# ============================================================
# 15. DUPLICATE FINANCIAL IMPACT
# ============================================================
# This is an impact assessment only.
# No records are removed here.


print("\n" + "=" * 70)
print("15. DUPLICATE FINANCIAL IMPACT")
print("=" * 70)


duplicate_rows = df.loc[
    duplicate_invoice_mask
].copy()

duplicate_rows["invoice_amount_numeric"] = (
    pd.to_numeric(
        duplicate_rows["invoice_amount"]
        .astype("string")
        .str.replace(",", "", regex=False)
        .str.strip(),
        errors="coerce"
    )
)

duplicate_invoice_value = (
    duplicate_rows["invoice_amount_numeric"]
    .sum()
)

print(
    f"\nInvoice amount represented by duplicate-group rows: "
    f"{duplicate_invoice_value:,.2f}"
)

print(
    "\nImportant:"
    "\nThis is not additional legitimate revenue."
    "\nIt represents financial value contained in records that"
    "\nparticipate in duplicate invoice groups."
)


# ============================================================
# 16. BUILD FINAL QUALITY REPORT
# ============================================================

quality_report = pd.DataFrame(issues)


# ------------------------------------------------------------
# Severity ordering
# ------------------------------------------------------------

severity_rank = {
    "CRITICAL": 1,
    "HIGH": 2,
    "MEDIUM": 3,
    "INVESTIGATE": 4,
    "EXPECTED": 5
}

quality_report["severity_rank"] = (
    quality_report["severity"]
    .map(severity_rank)
)

quality_report = (
    quality_report
    .sort_values(
        by=[
            "severity_rank",
            "affected_rows"
        ],
        ascending=[
            True,
            False
        ]
    )
    .drop(columns="severity_rank")
    .reset_index(drop=True)
)


# ============================================================
# 17. SEVERITY SUMMARY
# ============================================================

print("\n" + "=" * 70)
print("16. DATA-QUALITY SEVERITY SUMMARY")
print("=" * 70)

severity_summary = (
    quality_report["severity"]
    .value_counts()
    .reindex(
        [
            "CRITICAL",
            "HIGH",
            "MEDIUM",
            "INVESTIGATE",
            "EXPECTED"
        ],
        fill_value=0
    )
)

print(
    "\nNumber of audit rules by severity:"
)

print(severity_summary)


# ============================================================
# 18. FINAL DATA QUALITY REPORT
# ============================================================

print("\n" + "=" * 70)
print("17. FINAL DATA QUALITY REPORT")
print("=" * 70)

print(
    quality_report[
        [
            "issue_id",
            "category",
            "rule",
            "affected_rows",
            "affected_pct",
            "severity"
        ]
    ].to_string(index=False)
)


# ============================================================
# 19. SAVE DATA-QUALITY REPORT
# ============================================================
# This creates a standalone audit artifact.
#
# The raw CSV remains untouched.


output_path = Path(
    r"C:\Users\hp\Downloads\Report.csv"
)

quality_report.to_csv(
    output_path,
    index=False
)

print("\n" + "=" * 70)
print("AUDIT COMPLETE")
print("=" * 70)

print(
    f"\nData-quality report saved to:\n{output_path}"
)

print(
    "\nRaw dataset was NOT modified."
)


# ============================================================
# 20. FINAL AUDIT REMINDER
# ============================================================
# DO NOT sum affected_rows across different rules to estimate
# the number of bad records.
#
# One record can violate multiple rules.
#
# We therefore report issue counts per rule rather than creating
# a misleading total such as "24,009 bad rows."


print("\n" + "=" * 70)
print("PHASE 2 COMPLETED")
print("=" * 70)

print(
    "\nThe audit has classified the raw-data issues."
)

print(
    "Phase 3 will perform the actual cleaning and transformation."
)

PHASE 2 — DATA QUALITY AUDIT

Dataset shape: (12110, 14)
Rows: 12,110
Columns: 14

6. MISSING-VALUE AUDIT

Open invoices missing payment_date: 2,177
Open invoices missing payment_amount: 2,165
Paid invoices missing payment_date: 0
Paid invoices missing payment_amount: 15
Missing invoice_amount: 30
Missing customer_id: 144
Missing region: 217
Missing payment_method: 180

7. DUPLICATE-RECORD AUDIT

Rows belonging to exact duplicate groups: 219
Excess exact duplicate rows: 110
Unique duplicated invoice IDs: 109
Rows associated with duplicated invoice IDs: 219
Excess duplicate invoice rows: 110

8. KEY-INTEGRITY AUDIT

Missing invoice_id: 0
Excess duplicate invoice_id rows: 110
Missing supplier_id: 0
Missing product_id: 0

9. NUMERIC-INTEGRITY AUDIT

Present but non-numeric invoice_amount values: 80
Negative invoice amounts: 81
Negative payment amounts: 23
Zero payment amounts: 17

10. DATE-INTEGRITY AUDIT

invoice_date
  Missing: 0
  Present but invalid: 70

due_date
  Missing: 0
  Presen